In [2]:
from sklearn.metrics import accuracy_score, confusion_matrix

In [3]:
from speechbrain.inference import EncoderClassifier
from speechbrain.utils.fetching import LocalStrategy

classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_models/spkrec-ecapa-voxceleb",
    local_strategy=LocalStrategy.COPY
)

In [4]:
signal = classifier.load_audio("../data/processed/train/Sungho_0.wav")
embedding = classifier.encode_batch(signal)
print(embedding.shape)

torch.Size([1, 1, 192])


In [5]:
def extract_embedding(file_path):
    signal = classifier.load_audio(file_path)
    embedding = classifier.encode_batch(signal)
    embedding = embedding.squeeze().numpy()
    return embedding

In [6]:
import os
import numpy as np

train_folder = "../data/processed/train/"
train_files = [f for f in os.listdir(train_folder) if f.endswith(".wav")]

X_train_emb_list = []
y_train_emb_list = []

for filename in train_files:
    member_name = filename.split("_")[0]
    y_train_emb_list.append(member_name)
    
    file_path = train_folder + filename
    emb = extract_embedding(file_path)
    X_train_emb_list.append(emb)

X_train_emb = np.array(X_train_emb_list)
y_train_emb = np.array(y_train_emb_list)

print(X_train_emb.shape)
print(y_train_emb.shape)

(48, 192)
(48,)


test file

In [7]:
test_folder = "../data/processed/test/"
test_files = [f for f in os.listdir(test_folder) if f.endswith(".wav")]

X_test_emb_list = []
y_test_emb_list = []

for filename in test_files:
    member_name = filename.split("_")[0]
    y_test_emb_list.append(member_name)
    
    file_path = test_folder + filename
    emb = extract_embedding(file_path)
    X_test_emb_list.append(emb)

X_test_emb = np.array(X_test_emb_list)
y_test_emb = np.array(y_test_emb_list)

print(X_test_emb.shape)
print(y_test_emb.shape)

(14, 192)
(14,)


In [8]:
member_list_emb = sorted(set(y_train_emb))
centroids = []

for member in member_list_emb:
    indices = [i for i, label in enumerate(y_train_emb) if label == member]
    centroid = X_train_emb[indices].mean(axis=0)
    centroids.append(centroid)

centroids = np.array(centroids)
print(centroids.shape)

(6, 192)


Nearest Centroid Classifier used.... Test1.... accuracy 78

In [9]:
from scipy.spatial.distance import cdist

distances = cdist(X_test_emb, centroids, metric='euclidean')
predicted_indices = np.argmin(distances, axis=1)
y_pred_centroid = np.array([member_list_emb[i] for i in predicted_indices])

print("Accuracy:", accuracy_score(y_test_emb, y_pred_centroid))
print(confusion_matrix(y_test_emb, y_pred_centroid, labels=member_list_emb))

Accuracy: 0.7857142857142857
[[2 0 0 0 0 1]
 [0 2 0 0 0 0]
 [0 0 2 0 0 0]
 [0 0 0 2 0 0]
 [0 0 0 1 2 0]
 [1 0 0 0 0 1]]


Woonhakla Jaehyunu qarisiq salir---- ortaq featurelere baxmaq olar
Taesanda tek problem bir defe sungho ile qarisiq salmasidi

In [11]:
from sklearn.metrics import classification_report

print(classification_report(y_test_emb, y_pred_centroid))

              precision    recall  f1-score   support

     Jaehyun       0.67      0.67      0.67         3
      Leehan       1.00      1.00      1.00         2
       Riwoo       1.00      1.00      1.00         2
      Sungho       0.67      1.00      0.80         2
      Taesan       1.00      0.67      0.80         3
     Woonhak       0.50      0.50      0.50         2

    accuracy                           0.79        14
   macro avg       0.81      0.81      0.79        14
weighted avg       0.81      0.79      0.79        14



In [15]:
riwoo_idx = member_list_emb.index("Riwoo")
riwoo_c = centroids[riwoo_idx]
others_centroids = centroids[[i for i in range(len(member_list_emb)) if i != riwoo_idx]]
others_names = [m for m in member_list_emb if m != "Riwoo"]

# Riwoo-nun, hər ölçüdə, "digərlərin ortalamasından" nə qədər uzaq olduğu
others_mean = others_centroids.mean(axis=0)
riwoo_diff_from_others = np.abs(riwoo_c - others_mean)

# Digərlərin, öz aralarında (Riwoo-suz), hər ölçüdə, nə qədər dəyişkən olduğu
others_spread = others_centroids.std(axis=0)

# "Riwoo-ya xas" ölçü = Riwoo-nun fərqi BÖYÜK, amma, digərlərin öz-içi dəyişkənliyi KİÇİK
riwoo_uniqueness = riwoo_diff_from_others / (others_spread + 1e-6)  # sıfıra bölünməni önləmək üçün kiçik ədəd

top_unique_idx = np.argsort(riwoo_uniqueness)[::-1][:10]
print("Riwoo-ya ən çox xas olan ölçülər:", top_unique_idx)
print("Uniqueness skoru:", riwoo_uniqueness[top_unique_idx])

Riwoo-ya ən çox xas olan ölçülər: [182 174 178  79  77 148  90 151  65  36]
Uniqueness skoru: [7.687907  6.4105988 5.844224  5.474512  5.1974416 4.9153914 4.799673
 4.711558  4.5279937 4.4993753]


In [16]:
for target in member_list_emb:
    target_idx = member_list_emb.index(target)
    target_c = centroids[target_idx]
    others_centroids = centroids[[i for i in range(len(member_list_emb)) if i != target_idx]]
    
    others_mean = others_centroids.mean(axis=0)
    target_diff_from_others = np.abs(target_c - others_mean)
    others_spread = others_centroids.std(axis=0)
    
    uniqueness = target_diff_from_others / (others_spread + 1e-6)
    top_unique_idx = np.argsort(uniqueness)[::-1][:5]
    
    print(f"--- {target} ---")
    print("Ən xas ölçülər:", top_unique_idx)
    print("Uniqueness skoru:", uniqueness[top_unique_idx])
    print()

--- Jaehyun ---
Ən xas ölçülər: [ 59  49 139  84  24]
Uniqueness skoru: [8.181044  6.6103125 4.0448556 3.943826  3.7727304]

--- Leehan ---
Ən xas ölçülər: [ 39 165 107 161  92]
Uniqueness skoru: [8.177156  6.6584764 5.9055963 5.1173353 5.0899434]

--- Riwoo ---
Ən xas ölçülər: [182 174 178  79  77]
Uniqueness skoru: [7.687907  6.4105988 5.844224  5.474512  5.1974416]

--- Sungho ---
Ən xas ölçülər: [124 187   4 143 121]
Uniqueness skoru: [4.706734  4.689026  3.4692357 3.3624988 3.2290342]

--- Taesan ---
Ən xas ölçülər: [ 81  82  62 132  60]
Uniqueness skoru: [5.6291957 5.089065  4.521588  3.8691564 3.3444672]

--- Woonhak ---
Ən xas ölçülər: [94 29 38 40 76]
Uniqueness skoru: [9.403871  6.0948358 4.7557445 3.988602  3.2771645]

